In [5]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import json

import numpy as np
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_trf")

encode_bit_1 = ["VERB", "ADP", "SCONJ", "DET", "PRON", "ADJ", "PUNCT"]
encode_bit_0 = ["ADV", "AUX", "PART", "PROPN", "NOUN", "CCONJ", "NUM"]
neutral_pos = ["INTJ", "SYM"]

aux_token = ["aren", "wer", "weren", "wasn", "doesn", "didn", "couldn", "shan", "cannot", "didnt", "dont", "don", "youre", "im", "theyre"]


def in_aux_token(token):
    return token.lower() in aux_token

In [6]:
def load_corpus(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)


CORPUS_PATH = "../../results/generated_corpus.jsonl"
df = load_corpus(CORPUS_PATH)
stego = df[df["label"] == "stego"].reset_index(drop=True)
print(f"Stego texts: {len(stego)}")

Stego texts: 272


In [7]:
def encoding_part_stats(prompt, text, hidden_message):
    doc = nlp(prompt + text)
    prompt_len = len(prompt)
    exploded = []
    i = 0
    while i < len(doc):
        tok = doc[i]
        if tok.text.lower() == "can" and i + 1 < len(doc) and doc[i + 1].text.lower() == "not":
            exploded.append(("cannot", "VERB", tok.idx))
            i += 2
            continue
        if i + 1 < len(doc) and doc[i + 1].text in {"n't", "n’t"}:
            exploded.append((tok.text + "n", tok.pos_, tok.idx))
            exploded.append(("'t", "PART", doc[i + 1].idx))
            i += 2
            continue
        if tok.text.lower() in {"you", "they"} and i + 1 < len(doc) and doc[i + 1].text.lower() in {"re"}:
            exploded.append((tok.text + doc[i + 1].text, "VERB", tok.idx))
            i += 2
            continue
        exploded.append((tok.text, tok.pos_, tok.idx))
        i += 1

    generated = [(w, p, idx) for w, p, idx in exploded if idx >= prompt_len and p != "SPACE" and w.strip()]

    decoded = ""
    last_pos, last_text = None, ""
    end_idx = None
    for word, pos, idx in exploded:
        if in_aux_token(word):
            pos = "VERB"
        if idx < prompt_len:
            last_pos, last_text = pos, word
            continue
        if len(decoded) == len(hidden_message):
            break
        if last_pos == "PROPN" and pos == "PROPN" and last_text[:1].isupper() and word[:1].isupper():
            pass
        elif word in {"'s", "’s", "'t", "’t"}:
            pass
        elif pos in encode_bit_0:
            decoded += "0"
        elif pos in encode_bit_1:
            decoded += "1"
        last_pos, last_text = pos, word
        end_idx = idx

    encoding_part = [g for g in generated if g[2] <= end_idx]
    return {"bits": len(hidden_message), "tokens_encoding": len(encoding_part), "tokens_total": len(generated)}

In [8]:
stats = pd.DataFrame([encoding_part_stats(r.prompt, r.text, r.hidden_message) for r in stego.itertuples()])

bit_token_share = stats["bits"].sum() / stats["tokens_encoding"].sum()
capacity_encoding = (stats["bits"] / stats["tokens_encoding"]).mean()
capacity_total = (stats["bits"] / stats["tokens_total"]).mean()

print(f"Share of bit-carrying words/punctuation marks in the encoding part: {100 * bit_token_share:.1f}%")
print(f"Capacity of the encoding part: {capacity_encoding:.3f} bits per word or punctuation mark")
print(f"Capacity including the generated ending: {capacity_total:.3f} bits per word or punctuation mark")

Share of bit-carrying words/punctuation marks in the encoding part: 99.2%
Capacity of the encoding part: 0.992 bits per word or punctuation mark
Capacity including the generated ending: 0.725 bits per word or punctuation mark
